In [ ]:
!pip install -q unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 17.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import torch
from google.colab import drive
from unsloth import FastLanguageModel
from transformers import logging
from sklearn.metrics import f1_score, accuracy_score, classification_report
logging.set_verbosity_error()

drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/assetmind"
df = pd.read_parquet(f"{BASE}/phmsa_cause.parquet")

# Loads the base model and applies the trained adapter saved in Drive
model, tok = FastLanguageModel.from_pretrained(f"{BASE}/qwen3_phmsa_lora", max_seq_length=2048, load_in_4bit=True)
FastLanguageModel.for_inference(model)
tok.padding_side = "left"

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
Mounted at /content/drive
==((====))==  Unsloth 2026.10.7: Fast Qwen3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [ ]:
# ---- Prompt v1 (identical to training) ----
CAUSES = {
    "A": ("CORROSION FAILURE", "metal loss from internal or external corrosion, e.g. pitting, wall thinning"),
    "B": ("EQUIPMENT FAILURE", "malfunction or failure of equipment: pumps, valves, seals, gaskets, threaded or "
                               "non-threaded connections, control or relief equipment"),
    "C": ("EXCAVATION DAMAGE", "damage from digging or excavation activity by the operator or a third party"),
    "D": ("INCORRECT OPERATION", "human error: wrong valve position, overfilling, not following procedures, "
                                 "improper installation or operation"),
    "E": ("MATERIAL FAILURE OF PIPE OR WELD", "defects or cracking in the pipe body, seam or weld, from "
                                              "manufacturing, construction or environmental cracking"),
    "F": ("NATURAL FORCE DAMAGE", "earth movement, heavy rain or flooding, lightning, freezing temperatures, high winds"),
    "G": ("OTHER OUTSIDE FORCE DAMAGE", "external damage not from excavation: vehicles, fire or explosion, "
                                        "boats, intentional damage, previous mechanical damage"),
}
cause_list = "\n".join(f"{k}. {name}: {desc}" for k, (name, desc) in CAUSES.items())


def cause_prompt(text):
    return [{"role": "system", "content": "You are a pipeline integrity engineer classifying incident reports."},
            {"role": "user", "content":
                f"Classify the cause of this pipeline incident into exactly one category.\n\n{cause_list}\n\n"
                "If the narrative contains a supplemental or final update, use the cause stated there.\n\n"
                f"Narrative:\n{text}\n\nAnswer with a single letter (A-G) only."}]


# ---- Batched generation: runs prompts through the model 8 at a time ----
def generate(message_lists, max_new_tokens=16, batch_size=8):
    outputs = []
    for i in range(0, len(message_lists), batch_size):
        prompts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
                   for m in message_lists[i:i + batch_size]]
        enc = tok(prompts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)
        outputs += tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        if (i // batch_size) % 20 == 0:
            print(f"{min(i + batch_size, len(message_lists))}/{len(message_lists)}")
    return [o.strip() for o in outputs]


# ---- Parse the model's answer into a cause name ----
def to_cause(answer):
    letter = next((ch for ch in answer.upper() if ch in CAUSES), None)
    return CAUSES[letter][0] if letter else "INVALID"

In [ ]:
val = df[(df["split"] == "val") & df["main_track"]]
sample_answers = generate([cause_prompt(t) for t in val["text"].head(8)], max_new_tokens=16)
for a, label in zip(sample_answers, val["label"].head(8)):
    print(repr(a), "→", to_cause(a), "| true:", label)

8/8
'<think>\n\n</think>\n\nA' → CORROSION FAILURE | true: CORROSION FAILURE
'<think>\n\n</think>\n\nA' → CORROSION FAILURE | true: CORROSION FAILURE
'<think>\n\n</think>\n\nB' → EQUIPMENT FAILURE | true: EQUIPMENT FAILURE
'<think>\n\n</think>\n\nA' → CORROSION FAILURE | true: CORROSION FAILURE
'<think>\n\n</think>\n\nA' → CORROSION FAILURE | true: CORROSION FAILURE
'<think>\n\n</think>\n\nA' → CORROSION FAILURE | true: EQUIPMENT FAILURE
'<think>\n\n</think>\n\nB' → EQUIPMENT FAILURE | true: EQUIPMENT FAILURE
'<think>\n\n</think>\n\nB' → EQUIPMENT FAILURE | true: EQUIPMENT FAILURE


In [ ]:
val_pred = [to_cause(a) for a in generate([cause_prompt(t) for t in val["text"]], max_new_tokens=16)]
print("Invalid answers:", sum(p == "INVALID" for p in val_pred))
print(f"Fine-tuned validation macro-F1 {f1_score(val['label'], val_pred, average='macro'):.3f}, "
      f"accuracy {accuracy_score(val['label'], val_pred):.3f}")
print(classification_report(val["label"], val_pred, zero_division=0))

8/590
168/590
328/590
488/590
Invalid answers: 0
Fine-tuned validation macro-F1 0.801, accuracy 0.875
                                  precision    recall  f1-score   support

               CORROSION FAILURE       0.88      0.96      0.92       153
               EQUIPMENT FAILURE       0.91      0.91      0.91       265
               EXCAVATION DAMAGE       0.87      0.93      0.90        14
             INCORRECT OPERATION       0.86      0.83      0.84        82
MATERIAL FAILURE OF PIPE OR WELD       0.74      0.56      0.64        41
            NATURAL FORCE DAMAGE       0.78      0.74      0.76        19
      OTHER OUTSIDE FORCE DAMAGE       0.75      0.56      0.64        16

                        accuracy                           0.87       590
                       macro avg       0.83      0.78      0.80       590
                    weighted avg       0.87      0.87      0.87       590



In [ ]:
import time

test_main = df[(df["split"] == "test") & df["main_track"]]
t0 = time.time()
test_pred = [to_cause(a) for a in generate([cause_prompt(t) for t in test_main["text"]])]
elapsed = time.time() - t0

print("Invalid answers:", sum(p == "INVALID" for p in test_pred))
print(f"TEST macro-F1 {f1_score(test_main['label'], test_pred, average='macro'):.3f}, "
      f"accuracy {accuracy_score(test_main['label'], test_pred):.3f}")
print(f"Speed: {elapsed / len(test_main):.2f} s per narrative on a T4")
print(classification_report(test_main["label"], test_pred, zero_division=0))

pd.DataFrame({"label": test_main["label"].values, "pred": test_pred}, index=test_main.index).to_parquet(
    f"{BASE}/phmsa_main_finetuned_test.parquet")

8/742
168/742
328/742
488/742
648/742
Invalid answers: 0
TEST macro-F1 0.841, accuracy 0.881
Speed: 0.58 s per narrative on a T4
                                  precision    recall  f1-score   support

               CORROSION FAILURE       0.94      0.93      0.93       204
               EQUIPMENT FAILURE       0.91      0.89      0.90       353
               EXCAVATION DAMAGE       1.00      0.79      0.88        19
             INCORRECT OPERATION       0.69      0.88      0.78        69
MATERIAL FAILURE OF PIPE OR WELD       0.80      0.77      0.79        48
            NATURAL FORCE DAMAGE       0.90      0.68      0.78        28
      OTHER OUTSIDE FORCE DAMAGE       0.82      0.86      0.84        21

                        accuracy                           0.88       742
                       macro avg       0.87      0.83      0.84       742
                    weighted avg       0.89      0.88      0.88       742



In [ ]:
from sklearn.metrics import precision_score, recall_score

def pending_prompt(text):
    return [{"role": "system", "content": "You are a pipeline integrity engineer reviewing incident reports."},
            {"role": "user", "content":
                "Has the cause of this incident been determined?\n"
                "Answer DETERMINED if the narrative states a cause, including in a later supplemental or final update.\n"
                "Answer UNDETERMINED only if the cause is still under investigation and no later conclusion is given.\n\n"
                f"Narrative:\n{text}\n\nAnswer with one word: DETERMINED or UNDETERMINED."}]

test_all = df[df["split"] == "test"]
answers = generate([pending_prompt(t) for t in test_all["text"]], max_new_tokens=8)
pend_pred = ["UNDETERMINED" in a.upper() for a in answers]
print("Unclear answers:", sum("DETERMINED" not in a.upper() for a in answers))
print(f"Pending (fine-tuned model): precision {precision_score(test_all['pending'], pend_pred, zero_division=0):.3f}, "
      f"recall {recall_score(test_all['pending'], pend_pred, zero_division=0):.3f}, "
      f"F1 {f1_score(test_all['pending'], pend_pred, zero_division=0):.3f}")

8/778
168/778
328/778
488/778
648/778
Unclear answers: 59
Pending (fine-tuned model): precision 0.000, recall 0.000, F1 0.000
